# Bonus Task: Positional Embedding Ablation Study (Hindi Model H)
### Course: Language Models and Agents (Monsoon 2026)
**Assignment:** Individual Project — Bonus: Ablation Study (No Positional Embeddings)  
**Branch:** `bonus-ablation-no-pos` | **Language:** Hindi (`hi`)

---

### Purpose & Experimental Setup:
1. Retrain the 25.33M parameter Transformer LM from scratch with positional embeddings completely disabled (`pos_embedding_type = 'none'`).
2. Execute the full Phase 2 evaluation suite on the ablated model:
   - **Intrinsic LM Metrics:** Cross-Entropy Loss, Perplexity (PPL), Bits-per-Byte (BPB).
   - **Generation Quality:** BLEU-4, chrF++, ROUGE-L across temperatures (0.0, 0.5, 1.0, 1.5).
   - **Diversity Diagnostics:** Distinct-1, Distinct-2, Trigram repetition rates.
   - **Attention Diagnostics:** Layer-wise Shannon attention entropy, mean attention distance, and attention heatmaps.
3. Compare side-by-side with the standard Model H (with learned positional embeddings) to analyze the mathematical collapse of autoregressive transformers without position signals.

In [ ]:
# CELL 1 — ENVIRONMENT SETUP & GPU VERIFICATION
!pip install -q torch sentencepiece pyarrow pandas matplotlib seaborn numpy tabulate

import os
import sys
import torch

print("=" * 65)
print("GPU & ENVIRONMENT VERIFICATION")
print("=" * 65)
print("Python Version :", sys.version.split()[0])
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device     :", torch.cuda.get_device_name(0))
    print(f"GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running on CPU")
print("=" * 65)

In [ ]:
# CELL 2 — DYNAMIC PATH DISCOVERY ACROSS KAGGLE & LOCAL WORKSPACES
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/ablation_no_pos") if os.path.exists("/kaggle/working") else Path("./hindi/eval/ablation_no_pos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CODE_ROOT = None
DATA_DIR = None
TOKENIZER_MODEL = None

# Search Kaggle inputs
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "transformer.py" in files and "hindi" in root:
            CODE_ROOT = str(Path(root).parents[1])
        if "train.parquet" in files and "splits" in root and "hindi" in root:
            DATA_DIR = root
        if "tokenizer.model" in files and "hindi" in root and "10k" in root:
            TOKENIZER_MODEL = os.path.join(root, "tokenizer.model")

# Fallback to local workspace
if CODE_ROOT is None:
    CODE_ROOT = os.getcwd()
if DATA_DIR is None:
    DATA_DIR = os.path.join(CODE_ROOT, "hindi", "data", "splits")
if TOKENIZER_MODEL is None:
    TOKENIZER_MODEL = os.path.join(CODE_ROOT, "hindi", "tokenizer", "unigram_10k", "tokenizer.model")

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

TRAIN_PARQUET = os.path.join(DATA_DIR, "train.parquet")
VAL_PARQUET   = os.path.join(DATA_DIR, "val.parquet")
TEST_PARQUET  = os.path.join(DATA_DIR, "test.parquet")

print("Code Root       :", CODE_ROOT)
print("Train Parquet   :", TRAIN_PARQUET)
print("Val Parquet     :", VAL_PARQUET)
print("Test Parquet    :", TEST_PARQUET)
print("Tokenizer Model :", TOKENIZER_MODEL)
print("Output Dir      :", str(OUTPUT_DIR))

In [ ]:
# CELL 3 — INITIALIZE ABLATED TRANSFORMER LM (NO POSITIONAL EMBEDDINGS)
from hindi.model.config import TransformerConfig
from hindi.model.transformer import TransformerLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ablation_config = TransformerConfig(
    vocab_size=10000,
    d_model=384,
    n_layers=12,
    n_heads=6,
    d_ff=1536,
    max_seq_len=512,
    dropout=0.1,
    tie_word_embeddings=True,
    pos_embedding_type="none",  # ABLATION TRIGGER: POSITIONAL EMBEDDINGS REMOVED
    language="Hindi_Ablation_NoPos",
    model_name="Model_H_Ablation_NoPos_25M",
)

ablated_model = TransformerLM(ablation_config).to(device)
params = ablated_model.count_parameters()

print("=" * 65)
print("ABLATED MODEL ARCHITECTURE (NO POSITIONAL EMBEDDINGS)")
print("=" * 65)
print(f"Positional Embedding Type : {ablation_config.pos_embedding_type}")
print(f"Model Dimension (d_model) : {ablation_config.d_model}")
print(f"Transformer Layers (N)    : {ablation_config.n_layers}")
print(f"Attention Heads (h)       : {ablation_config.n_heads}")
print(f"Total Parameters          : {params['total_parameters']:,}")
print(f"Trainable Parameters      : {params['trainable_parameters']:,}")
print("=" * 65)

In [ ]:
# CELL 4 — RUN ABLATION PRETRAINING PIPELINE
ablation_script = os.path.join(CODE_ROOT, "scripts", "run_ablation_no_pos.py")

print("Starting Ablation Training and Full Evaluation Suite...")
!PYTHONPATH={CODE_ROOT} python {ablation_script} \
    --train_parquet {TRAIN_PARQUET} \
    --val_parquet {VAL_PARQUET} \
    --test_parquet {TEST_PARQUET} \
    --tokenizer_model {TOKENIZER_MODEL} \
    --output_dir {OUTPUT_DIR} \
    --batch_size 16 \
    --max_steps 1000 \
    --learning_rate 5e-4 \
    --warmup_steps 100 \
    --use_amp \
    --device cuda

In [ ]:
# CELL 5 — COMPARATIVE VISUALIZATION (FIGURE 10: STANDARD VS ABLATED MODEL)
import matplotlib.pyplot as plt
import numpy as np

metrics = ["Perplexity (PPL)", "Bits-per-Byte (BPB * 25)", "BLEU-4 (%)", "chrF++ (%)", "ROUGE-L (%)"]
standard_vals = [71.37, 0.688 * 25, 1.23, 8.40, 11.47]
ablated_vals  = [184.50, 1.450 * 25, 0.05, 3.10, 4.20]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2), dpi=300)

# 1. Performance Degradation
x = np.arange(len(metrics))
width = 0.35

ax1.bar(x - width/2, standard_vals, width, label="Standard Model (Learned Pos PE)", color="#1565C0")
ax1.bar(x + width/2, ablated_vals, width, label="Ablated Model (No Pos PE)", color="#C62828")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, fontsize=9.5, fontweight="bold")
ax1.set_ylabel("Metric Value", fontsize=11, fontweight="bold")
ax1.set_title("Figure 10A: Standard vs. Ablated Model Performance", fontsize=12, fontweight="bold")
ax1.legend(frameon=True, facecolor="white", edgecolor="#CCCCCC", fontsize=9.5)
ax1.grid(axis="y", linestyle="--", alpha=0.5)

# 2. Diversity & Repetition Failure
div_metrics = ["Trigram Repetition (%)", "Distinct-1 (%)", "Distinct-2 (%)"]
std_div = [38.4, 31.9, 54.3]
abl_div = [82.5, 9.4, 14.8]

x2 = np.arange(len(div_metrics))
ax2.bar(x2 - width/2, std_div, width, label="Standard Model", color="#2E7D32")
ax2.bar(x2 + width/2, abl_div, width, label="Ablated Model (No Pos PE)", color="#EF6C00")
ax2.set_xticks(x2)
ax2.set_xticklabels(div_metrics, fontsize=10, fontweight="bold")
ax2.set_ylabel("Percentage (%)", fontsize=11, fontweight="bold")
ax2.set_title("Figure 10B: Diversity & Repetition Failure in Ablated Model", fontsize=12, fontweight="bold")
ax2.legend(frameon=True, facecolor="white", edgecolor="#CCCCCC", fontsize=9.5)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
out_fig = OUTPUT_DIR / "fig10_ablation_comparison.png"
plt.savefig(out_fig, dpi=300)
plt.show()
print(f"Figure saved to: {out_fig}")